# Images are just arrays: a first look at image analysis

Sooner or later, most benchwork produces an image: a microscopy field of cells, a gel, a blot, a plate scan. And the single most important idea in this whole module is:

**A digital image is just a numpy array of numbers.**

That's it. That's the secret. Everything you already know about arrays - indexing, slicing, comparisons, histograms - works on images. By the end of this module you'll load a real fluorescence microscopy image, segment the nuclei in it, and *count and measure them* - the bread-and-butter quantification behind a huge fraction of cell biology papers.

We'll use **scikit-image**, the standard python imaging library. Install it (and napari, for the last section) with `mamba install -c conda-forge scikit-image napari pyqt` in your terminal.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage as ski

## Loading an image

scikit-image ships with real sample images so everyone has identical data. `human_mitosis` is a fluorescence microscopy image of human cells with their nuclei stained (DNA labeled, so nuclei glow):

In [ ]:
image = ski.data.human_mitosis()
image

See? An array. What are its dimensions and data type?

In [ ]:
print(image.shape)
print(image.dtype)
print(image.min(), image.max())

A 512 x 512 grid of pixels, each an integer from 0-255 (that's what `uint8` means - 8 bits per pixel). Row and column indices are positions; the *value* at each position is brightness.

(In your own work you'd load a file instead: `image = ski.io.imread('my_image.tif')` - same array, same everything from here on.)

Numbers are hard to stare at, so let's *look* at it with matplotlib's `imshow`:

In [ ]:
plt.imshow(image, cmap='gray')
plt.colorbar()

Cells! Each bright blob is one nucleus. A couple of very bright small ones are cells caught in mitosis (condensed DNA packs the signal into a smaller space).

The `cmap='gray'` argument is the *colormap* - the rule for turning numbers into colors. The detector didn't record color, just intensity, so the colormap is our choice. Try changing it to `'viridis'` or `'magma'` and re-running.

Because an image is an array, cropping is just **slicing** - `[rows, columns]`, exactly like Day 3:

In [ ]:
plt.imshow(image[300:400, 100:200], cmap='gray')
plt.colorbar()

And reading out a single pixel's brightness is just indexing:

In [ ]:
image[350, 150]

## The histogram: an image's fingerprint

Before analyzing, look at the *distribution* of pixel values. `.ravel()` flattens the 2D array into 1D so we can histogram it:

In [ ]:
plt.hist(image.ravel(), bins=50)
plt.xlabel('pixel intensity')
plt.ylabel('number of pixels')
plt.yscale('log')

Two populations: a huge dark peak on the left (background - most of the image is empty space between cells) and a long bright tail on the right (the nuclei). Keep this picture in mind - it's about to become our segmentation strategy.

## Segmentation: which pixels are cells?

**Segmentation** means deciding which pixels belong to the objects you care about. Looking at that histogram, the obvious approach: pick a cutoff intensity, and call everything brighter than it "nucleus".

We could eyeball a cutoff, but there's a classic algorithm - **Otsu's method** - that picks the threshold separating the two populations optimally:

In [ ]:
threshold = ski.filters.threshold_otsu(image)
threshold

Now the fun part. Remember boolean comparisons on arrays? `image > threshold` compares *every pixel at once* and gives us a True/False array - our **mask**:

In [ ]:
mask = image > threshold

plt.imshow(mask, cmap='gray')

One line of numpy and we've separated cells from background. White = True = nucleus, black = False = background.

## From mask to *objects*: labeling

The mask knows which pixels are "nucleus-ish", but it doesn't know there are *separate* nuclei. `ski.measure.label` finds each connected white blob and gives it its own integer ID (blob 1, blob 2, ...):

In [ ]:
labels = ski.measure.label(mask)
labels.max()

Over 300 objects found! Let's see them - `label2rgb` gives every labeled object a random color so you can see the individual objects:

In [ ]:
plt.imshow(ski.color.label2rgb(labels, image=image, bg_label=0))

Pretty good! But look closely and you'll spot two classic problems:

1. Tiny specks of noise got counted as "nuclei"
2. Some touching nuclei got merged into one blob

Problem 1 is easy - throw out objects smaller than some area (a real nucleus here is hundreds of pixels):

In [ ]:
clean_mask = ski.morphology.remove_small_objects(mask, min_size=50)
labels = ski.measure.label(clean_mask)
labels.max()

Problem 2 (splitting touching nuclei) is the classic hard problem of cell biology image analysis - the standard answer is the *watershed* algorithm, and the modern answer is deep learning tools like Cellpose or StarDist. We'll happily ignore it today; our counts will be close enough.

## Quantification: measure everything

Counting is nice, but the real payoff is **measuring each object**. `regionprops_table` computes properties per labeled object and - even better - plays nicely with pandas:

In [ ]:
props = ski.measure.regionprops_table(
    labels, intensity_image=image,
    properties=['label', 'area', 'mean_intensity', 'eccentricity'])

nuclei = pd.DataFrame(props)
nuclei.head()

One row per nucleus, one column per measurement. From an image to a tidy DataFrame - and now we're back on completely familiar ground from Day 3 and 4. How big are nuclei in this image?

In [ ]:
plt.hist(nuclei['area'], bins=30)
plt.xlabel('nucleus area (pixels)')
plt.ylabel('count')

In [ ]:
nuclei['area'].describe()

(The little bump of very large "nuclei" is those merged blobs from problem 2.)

Is there anything interesting in the relationship between size and brightness?

In [ ]:
plt.plot(nuclei['area'], nuclei['mean_intensity'], 'o', alpha=0.5)
plt.xlabel('area (pixels)')
plt.ylabel('mean intensity')

See that little cloud of small-but-very-bright objects, separated from the main population? Those are the **mitotic cells** - condensed chromosomes concentrate the DNA stain into a small, intense blob. We just *rediscovered a biological subpopulation from a scatterplot*. This kind of thing - segment, measure, plot, notice - is image-based quantification in a nutshell.

One last real-world note: so far our areas are in *pixels*. Your microscope's metadata tells you the pixel size (e.g. 0.65 µm/pixel), and converting is just arithmetic: `area_um2 = area_px * 0.65**2`.

### Exercise:

1. Run the whole pipeline (threshold -> mask -> clean -> label -> count) on `ski.data.coins()` - a photo of coins on a dark background. How many coins does it find? Is the count right? (Look at the label image to debug - the background of that image is trickier!)
2. Back on `human_mitosis`: try to *isolate the mitotic cells* using the `nuclei` DataFrame - filter for small area and high mean intensity, and see how many you find.
3. What happens to your nucleus count if you smooth the image first with `ski.filters.gaussian(image, sigma=2, preserve_range=True)` before thresholding? Try a few `sigma` values. (Smoothing before thresholding is a very standard trick to suppress noise.)

## napari: a proper image viewer (runs on YOUR laptop only)

matplotlib is fine for small 2D images, but real microscopy data is big, multi-channel, 3D, and time-lapsed. **[napari](https://napari.org)** is a fast, python-native viewer built for exactly that - think "ImageJ/FIJI, but it speaks numpy".

**⚠️ Important:** napari opens a *desktop application window*. It works when you're running Jupyter **locally on your own machine** - it will NOT work on a remote JupyterHub/server, because the window would be opening on a computer in a data center somewhere. If you're on a remote setup today, just read along and try this at home.

The cells below are therefore *not* run as part of the module - they're your take-home dessert.

In [ ]:
# LOCAL ONLY - opens a separate window
import napari

viewer = napari.Viewer()
viewer.add_image(image, name="nuclei")

A window should have popped up (check behind your browser). Play with the contrast limits slider - unlike `imshow`, this is instant and interactive.

Now the killer feature: napari displays our *analysis results* as layers on top of the raw data - like ImageJ overlays, but straight from our numpy arrays:

In [ ]:
# LOCAL ONLY
viewer.add_labels(labels)

Toggle the layer's visibility (the eye icon), and mouse over nuclei - the object ID under your cursor shows in the corner. This is *the* way to sanity-check a segmentation: overlay it on the raw image and look.

napari really shines in 3D. scikit-image has a 3D two-channel confocal stack of cells (this one downloads ~8 MB the first time):

In [ ]:
# LOCAL ONLY (and needs internet the first time)
membranes, nuclei_3d = ski.data.cells3d().transpose(1, 0, 2, 3)

viewer3d = napari.Viewer()
viewer3d.add_image(nuclei_3d, name="nuclei", colormap="green", blending="additive")
viewer3d.add_image(membranes, name="membranes", colormap="magenta", blending="additive")

You get a z-slider through the stack for free - and if you click the 3D button (bottom-left of the napari window), you're flying around a volume-rendered stack of cells. Try doing *that* with matplotlib.

## Wrapping up

- An image is a numpy array; analysis = array operations you already know.
- **threshold -> mask -> label -> measure** is the fundamental quantification pipeline; what changes between papers is mostly *how fancy the segmentation step is*.
- For hard segmentation (touching cells, weird shapes): look up **Cellpose** and **StarDist** - pretrained deep learning models with friendly interfaces (and napari plugins!).
- napari + its [plugin ecosystem](https://www.napari-hub.org) covers a huge amount of day-to-day microscopy analysis without writing much code at all.